In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    cross_val_score,
    cross_val_predict
)

from sklearn.preprocessing import StandardScaler

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [4]:
df = pd.read_csv("/content/appointments_clean.csv")

print(df.shape)
print(df.head())

(990, 16)
  appointment_id patient_id clinic_id   booked_on   appt_date  age gender  \
0          A0001       P509       C05  2025-01-24  2025-01-25   60      F   
1          A0002       P046       C07  2025-05-27  2025-06-06   39      M   
2          A0003       P009       C02  2025-03-24  2025-03-30   63      F   
3          A0004       P482       C01  2025-04-18  2025-04-22   52      F   
4          A0005       P251       C01  2024-11-23  2025-01-02   64      F   

   distance_km  fee_inr booking_channel    visit_type reminder_sent  \
0         18.0      850           Phone  Consultation           Yes   
1         12.5      350         Walk-In     Follow-up            No   
2          4.1      850         Walk-In  Consultation            No   
3          7.5      750             App  Consultation           Yes   
4         19.0      700           Phone  Consultation            No   

   prior_no_shows    status  lead_days  no_show  
0               1   No-Show          1        1  


In [5]:
# Convert reminder_sent into numerical values
df["reminder_sent"] = df["reminder_sent"].map({
    "Yes": 1,
    "No": 0
})

# Select features
X = df[
    [
        "age",
        "distance_km",
        "fee_inr",
        "lead_days",
        "prior_no_shows",
        "reminder_sent",
        "booking_channel",
        "visit_type"
    ]
].copy()

# One-hot encoding
X = pd.get_dummies(
    X,
    columns=["booking_channel", "visit_type"]
)

# Target variable
y = df["no_show"]

print("Features:")
print(X.columns.tolist())

print("X shape:", X.shape)
print("y shape:", y.shape)

Features:
['age', 'distance_km', 'fee_inr', 'lead_days', 'prior_no_shows', 'reminder_sent', 'booking_channel_App', 'booking_channel_Phone', 'booking_channel_Walk-In', 'visit_type_Consultation', 'visit_type_Follow-up']
X shape: (990, 11)
y shape: (990,)


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [7]:
print("Training size:", len(X_train))
print("Testing size:", len(X_test))

print("No-shows in test set:", (y_test == 1).sum())

Training size: 792
Testing size: 198
No-shows in test set: 49


In [8]:
baseline_accuracy = (y_test == 0).mean()

print("Baseline Accuracy:", baseline_accuracy)

Baseline Accuracy: 0.7525252525252525


Task 3.2 — Logistic regression

In [9]:
scaler = StandardScaler()

X_train_s = scaler.fit_transform(X_train)

X_test_s = scaler.transform(X_test)

In [10]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train_s, y_train)
y_pred = model.predict(X_test_s)

y_prob = model.predict_proba(X_test_s)[:, 1]
model.predict_proba(X_test_s)

array([[0.97843458, 0.02156542],
       [0.84908341, 0.15091659],
       [0.48130818, 0.51869182],
       [0.80869967, 0.19130033],
       [0.83228205, 0.16771795],
       [0.68002282, 0.31997718],
       [0.66155211, 0.33844789],
       [0.63296216, 0.36703784],
       [0.89500102, 0.10499898],
       [0.82673934, 0.17326066],
       [0.26807994, 0.73192006],
       [0.96118775, 0.03881225],
       [0.87625299, 0.12374701],
       [0.92649755, 0.07350245],
       [0.89546229, 0.10453771],
       [0.96067678, 0.03932322],
       [0.85145772, 0.14854228],
       [0.46156982, 0.53843018],
       [0.95895024, 0.04104976],
       [0.45550513, 0.54449487],
       [0.8743726 , 0.1256274 ],
       [0.94582955, 0.05417045],
       [0.80529916, 0.19470084],
       [0.5535434 , 0.4464566 ],
       [0.07593398, 0.92406602],
       [0.81028145, 0.18971855],
       [0.96368209, 0.03631791],
       [0.61071219, 0.38928781],
       [0.97819932, 0.02180068],
       [0.80064131, 0.19935869],
       [0.

In [11]:
print("Accuracy:", accuracy_score(y_test, y_pred))

print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("Precision:", precision_score(y_test, y_pred, zero_division=0))

print("Recall:", recall_score(y_test, y_pred, zero_division=0))

print("F1 Score:", f1_score(y_test, y_pred, zero_division=0))

print("ROC-AUC:", roc_auc_score(y_test, y_prob))

Accuracy: 0.7878787878787878
Confusion Matrix:
[[139  10]
 [ 32  17]]
Precision: 0.6296296296296297
Recall: 0.3469387755102041
F1 Score: 0.4473684210526316
ROC-AUC: 0.788933022873579


In [14]:
coefficients = pd.DataFrame({
    "Feature": X.columns,
    "Coefficient": model.coef_[0]
})

print(coefficients)
coefficients["Odds_Ratio"] = np.exp(
    coefficients["Coefficient"]
)

                    Feature  Coefficient
0                       age    -0.253880
1               distance_km     0.253448
2                   fee_inr     0.452761
3                 lead_days     0.614471
4            prior_no_shows     0.649612
5             reminder_sent    -0.580669
6       booking_channel_App    -0.046804
7     booking_channel_Phone     0.215881
8   booking_channel_Walk-In    -0.186364
9   visit_type_Consultation     0.015658
10     visit_type_Follow-up    -0.015658


In [15]:
reminder_idx = X.columns.get_loc("reminder_sent")

reminder_coef = model.coef_[0][reminder_idx]

print("Scaled coefficient:", reminder_coef)

print("Exp of coefficient:", np.exp(reminder_coef))

Scaled coefficient: -0.5806686033659922
Exp of coefficient: 0.559524141750829


In [16]:
reminder_std = scaler.scale_[reminder_idx]

adjusted_coef = reminder_coef / reminder_std

reminder_odds_ratio = np.exp(adjusted_coef)

print("Exp of scaled coefficient:", np.exp(reminder_coef))

print("Adjusted coefficient:", adjusted_coef)

print("Odds ratio for reminder:", reminder_odds_ratio)

Exp of scaled coefficient: 0.559524141750829
Adjusted coefficient: -1.1731265130909097
Odds ratio for reminder: 0.3093980903124433


Task 3.3 — Decision tree and overfitting

In [17]:
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

train_pred = dt.predict(X_train)
test_pred = dt.predict(X_test)

print("Training Accuracy:", accuracy_score(y_train, train_pred))

print("Testing Accuracy:", accuracy_score(y_test, test_pred))

Training Accuracy: 1.0
Testing Accuracy: 0.6717171717171717


In [22]:
dt3 = DecisionTreeClassifier(
    max_depth=3,
    random_state=42
)

dt3.fit(X_train, y_train)

train_pred3 = dt3.predict(X_train)
test_pred3 = dt3.predict(X_test)

print("Depth-3 Training Accuracy:",
      accuracy_score(y_train, train_pred3))

print("Depth-3 Testing Accuracy:",
      accuracy_score(y_test, test_pred3))

Depth-3 Training Accuracy: 0.821969696969697
Depth-3 Testing Accuracy: 0.7777777777777778


Task 3.4 — Random forest and a comparison table

In [18]:
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [19]:
rf_pred = rf.predict(X_test)

rf_prob = rf.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, rf_pred))

print("Recall:", recall_score(y_test, rf_pred, zero_division=0))

print("ROC-AUC:", roc_auc_score(y_test, rf_prob))

Accuracy: 0.7474747474747475
Recall: 0.30612244897959184
ROC-AUC: 0.7057252431173813


In [20]:
importance_df = pd.DataFrame({
    "Feature": X.columns,
    "Importance": rf.feature_importances_
})

importance_df = importance_df.sort_values(
    by="Importance",
    ascending=False
)

print(importance_df.head(3))

       Feature  Importance
3    lead_days    0.219905
0          age    0.196788
1  distance_km    0.194739


In [23]:
dt3_prob = dt3.predict_proba(X_test)[:, 1]

In [24]:
comparison = pd.DataFrame([
    {
        "Model": "Logistic Regression",
        "Accuracy": accuracy_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, y_prob)
    },
    {
        "Model": "Decision Tree (Depth 3)",
        "Accuracy": accuracy_score(y_test, test_pred3),
        "Recall": recall_score(y_test, test_pred3, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, dt3_prob)
    },
    {
        "Model": "Random Forest",
        "Accuracy": accuracy_score(y_test, rf_pred),
        "Recall": recall_score(y_test, rf_pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, rf_prob)
    }
])

print(comparison.round(4))

                     Model  Accuracy  Recall  ROC_AUC
0      Logistic Regression    0.7879  0.3469   0.7889
1  Decision Tree (Depth 3)    0.7778  0.2245   0.6997
2            Random Forest    0.7475  0.3061   0.7057


Which model looks best and why; this table is for reporting — the formal choice is made in Task 3.5??\
**Random Forest may provide better predictive performance by combining multiple decision trees.**

Task 3.5 — Tune, choose the model and the threshold, then test once

In [25]:
RandomForestClassifier(n_estimators=100)

RandomForestClassifier()

In [26]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [3, 5, 8]
}

In [27]:
grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        "n_estimators": [100, 200],
        "max_depth": [3, 5, 8]
    },
    cv=5,
    scoring="roc_auc"
)

grid.fit(X_train, y_train)

print("Best Parameters:", grid.best_params_)

print("Best CV ROC-AUC:", grid.best_score_)

Best Parameters: {'max_depth': 3, 'n_estimators': 200}
Best CV ROC-AUC: 0.7673242251739405


In [28]:
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000))
])

In [29]:
lr_cv = cross_val_score(
    lr_pipe,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

dt3_cv = cross_val_score(
    dt3,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

rf_cv = cross_val_score(
    rf,
    X_train,
    y_train,
    cv=5,
    scoring="roc_auc"
).mean()

tuned_rf = grid.best_estimator_

tuned_rf_cv = grid.best_score_

In [30]:
cv_results = pd.DataFrame({
    "Model": [
        "Logistic Regression",
        "Decision Tree Depth 3",
        "Random Forest",
        "Tuned Random Forest"
    ],
    "CV_ROC_AUC": [
        lr_cv,
        dt3_cv,
        rf_cv,
        tuned_rf_cv
    ]
})

print(cv_results.sort_values(
    by="CV_ROC_AUC",
    ascending=False
))

                   Model  CV_ROC_AUC
0    Logistic Regression    0.773714
3    Tuned Random Forest    0.767324
2          Random Forest    0.746602
1  Decision Tree Depth 3    0.678449


In [31]:
models = {
    "Logistic Regression": lr_pipe,
    "Decision Tree Depth 3": dt3,
    "Random Forest": rf,
    "Tuned Random Forest": tuned_rf
}

cv_scores = {
    "Logistic Regression": lr_cv,
    "Decision Tree Depth 3": dt3_cv,
    "Random Forest": rf_cv,
    "Tuned Random Forest": tuned_rf_cv
}

best_model_name = max(cv_scores, key=cv_scores.get)

best_model = models[best_model_name]

print("Selected Model:", best_model_name)
print("Best CV ROC-AUC:", cv_scores[best_model_name])

Selected Model: Logistic Regression
Best CV ROC-AUC: 0.7737139899948249


In [32]:
oof_prob = cross_val_predict(
    best_model,
    X_train,
    y_train,
    cv=5,
    method="predict_proba"
)[:, 1]

In [33]:
threshold_results = []

for threshold in [0.5, 0.35]:

    oof_pred = (oof_prob >= threshold).astype(int)

    precision = precision_score(
        y_train,
        oof_pred,
        zero_division=0
    )

    recall = recall_score(
        y_train,
        oof_pred,
        zero_division=0
    )

    f1 = f1_score(
        y_train,
        oof_pred,
        zero_division=0
    )

    flagged = oof_pred.sum()

    threshold_results.append({
        "Threshold": threshold,
        "Precision": precision,
        "Recall": recall,
        "F1": f1,
        "Number_Flagged": flagged
    })

threshold_df = pd.DataFrame(threshold_results)

print(threshold_df.round(4))

   Threshold  Precision  Recall      F1  Number_Flagged
0       0.50     0.5753  0.2485  0.3471              73
1       0.35     0.5130  0.4675  0.4892             154


In [34]:
best_threshold = threshold_df.loc[
    threshold_df["F1"].idxmax(),
    "Threshold"
]

print("Selected Threshold:", best_threshold)

Selected Threshold: 0.35


In [35]:
best_model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('model', LogisticRegression(max_iter=1000))])

In [36]:
test_probs = best_model.predict_proba(X_test)[:, 1]

In [37]:
final_predictions = (
    test_probs >= best_threshold
).astype(int)

In [ ]:
print("Final Model:", best_model_name)

print(
    "Precision:",
    precision_score(y_test, final_predictions, zero_division=0)
)

print(
    "Recall:",
    recall_score(y_test, final_predictions, zero_division=0)
)

print("Number Flagged:", final_predictions.sum())

print("ROC-AUC:", roc_auc_score(y_test, test_probs))

In [38]:
risk_scores = pd.DataFrame({
    "appointment_id": df.loc[
        X_test.index, "appointment_id"
    ].values,

    "no_show": y_test.values,

    "risk_prob": test_probs,

    "flag": final_predictions
})

In [39]:
risk_scores.to_csv(
    "test_risk_scores.csv",
    index=False
)

print(risk_scores.head())

  appointment_id  no_show  risk_prob  flag
0          A0217        0   0.021565     0
1          A0336        0   0.150917     0
2          A0508        0   0.518692     1
3          A0314        0   0.191300     0
4          A0089        0   0.167718     0
